# Spheroid Segmentation — All 8 Models

This notebook trains and evaluates **all 8 segmentation architectures** used in the SLiMIA-IPP paper:

| Model | Backbone | Loss |
|-------|----------|------|
| U-Net++ | ResNet-50 | Focal Tversky |
| DeepLabV3 | ResNet-34 | BCE + Dice |
| DeepLabV3+ | ResNet-50 | Focal Tversky |
| Attention U-Net | ResNet-based | BCE + Dice |
| Swin-UNet | Swin-Tiny | BCEWithLogits |
| TransUNet | ViT + CNN | BCEWithLogits |
| RefineNet | ResNet-34 | Focal Tversky |
| SegNet | VGG-style | Focal Tversky |

All models use the **SLiMIA metadata CSV** for data loading (no folder walking).
Results are averaged across **3 random seeds** (42, 123, 999) for robustness.

> **Dataset:** [SLiMIA on Kaggle](https://www.kaggle.com/datasets/...)
> **Metadata CSV:** `data/slimia_metadata.csv` (included in repo)

In [ ]:
# Install required packages
# On Kaggle these are mostly pre-installed; uncomment if running locally
# !pip install segmentation-models-pytorch albumentations tifffile timm einops openpyxl tqdm scikit-learn

In [ ]:
# Imports
import os
import re
import random
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from timm.models.swin_transformer import swin_tiny_patch4_window7_224
from timm.models.vision_transformer import vit_base_patch16_224
from einops import rearrange

## Configuration

Set your CSV path here. The CSV should have at minimum two columns:
- `full_path` — absolute path to the `.ome.tiff` image
- `mask_path` — absolute path to the corresponding `.tiff` segmentation mask

Additional metadata columns (microscope, cell_line, etc.) are used by downstream IPP notebooks.

In [ ]:
# Configuration
"""
CSV Columns used here:
  full_path   : full path to the .ome.tiff image
  mask_path   : full path to the binary segmentation mask

The same CSV is used by all downstream IPP notebooks — it's the single
source of truth for data paths and metadata.
"""
CSV_PATH   = "../data/slimia_metadata.csv"   # update to your local path
CKPT_DIR   = "../checkpoints/segmentation/"
IMG_SIZE   = (256, 256)
BATCH_SIZE = 8
NUM_EPOCHS = 200
LR         = 1e-4
PATIENCE   = 40
SEEDS      = [42, 123, 999]               # run each model 3× and average
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")

In [ ]:
# Reproducibility
def set_seed(seed: int):
    """Pin all RNGs so results are reproducible across runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Dataset

In [ ]:
# Dataset (CSV-based)
class SLIMIADataset(Dataset):
    """
    Loads SLiMIA image-mask pairs from a pre-built metadata CSV.
    Grayscale images are expanded to 3 channels for ImageNet-pretrained backbones.
    Normalization uses per-image percentile clipping (p1/p99) which handles the
    wide intensity variation across different microscope types.
    """
    def __init__(self, csv_path: str, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.transform = transform
        print(f"Loaded {len(self.df)} samples from {csv_path}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = row["full_path"]
        mask_path = row["mask_path"]

        # Load
        image = tifffile.imread(img_path).astype(np.float32)
        mask  = tifffile.imread(mask_path).astype(np.float32)

        # Flatten multi-channel masks (take first channel if 3-D)
        if mask.ndim == 3:
            mask = mask[..., 0]

        # Percentile normalisation - robust to outlier pixels / bright artefacts
        p1, p99 = np.percentile(image, (1, 99))
        image   = np.clip(image, p1, p99)
        image   = (image - p1) / (p99 - p1 + 1e-8)
        image   = np.nan_to_num(image).astype(np.float32)

        # Binary mask
        mask = (mask > 0).astype(np.float32)

        # Grayscale - 3-channel (required by ImageNet-pretrained encoders)
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=-1)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"].unsqueeze(0).float()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask  = torch.from_numpy(mask).unsqueeze(0).float()

        return image, mask

In [ ]:
# Augmentations
# Standard heavy augmentation for most CNN-based models
train_transform = A.Compose([
    A.Resize(*IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomGamma(p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(*IMG_SIZE),
    ToTensorV2(),
])

# Transformer models (Swin-UNet, TransUNet) need 224×224
train_transform_224 = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    ToTensorV2(),
])

val_transform_224 = A.Compose([
    A.Resize(224, 224),
    ToTensorV2(),
])

## Losses & Metrics

In [ ]:
# Losses
class FocalTverskyLoss(nn.Module):
    """
    Focal Tversky loss. Used by U-Net++, DeepLabV3+, RefineNet, SegNet.
    α=0.7, β=0.3 weights FN more than FP — good for sparse foreground (spheroid cores).
    γ=0.75 applies focal modulation to hard examples.
    """
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma

    def forward(self, preds, targets):
        preds  = torch.sigmoid(preds)
        smooth = 1e-6
        TP = (preds * targets).sum(dim=(1, 2, 3))
        FP = ((1 - targets) * preds).sum(dim=(1, 2, 3))
        FN = (targets * (1 - preds)).sum(dim=(1, 2, 3))
        tversky = (TP + smooth) / (TP + self.alpha * FP + self.beta * FN + smooth)
        return ((1 - tversky) ** self.gamma).mean()


class BCEDiceLoss(nn.Module):
    """BCE + Dice combo. Used by DeepLabV3 and Attention U-Net."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        bce_loss  = self.bce(preds, targets)
        probs     = torch.sigmoid(preds)
        smooth    = 1e-6
        inter     = (probs * targets).sum(dim=(1, 2, 3))
        union     = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - ((2 * inter + smooth) / (union + smooth)).mean()
        return 0.5 * bce_loss + 0.5 * dice_loss


# Metrics
def dice_score(pred, target, threshold=0.5, eps=1e-6):
    pred  = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2 * inter + eps) / (union + eps)).mean()


def iou_score(pred, target, threshold=0.5, eps=1e-6):
    pred  = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = (pred + target - pred * target).sum(dim=(1, 2, 3))
    return ((inter + eps) / (union + eps)).mean()


def pixel_accuracy(pred, target, threshold=0.5):
    pred   = (torch.sigmoid(pred) > threshold).float()
    correct = (pred == target).float()
    return correct.mean()


def compute_all_metrics(pred, target, threshold=0.5, eps=1e-6):
    """Returns a dict with dice, iou, accuracy, precision, recall, f1."""
    probs = torch.sigmoid(pred)
    preds = (probs > threshold).float()
    TP = (preds * target).sum().item()
    TN = ((1 - preds) * (1 - target)).sum().item()
    FP = (preds * (1 - target)).sum().item()
    FN = ((1 - preds) * target).sum().item()
    accuracy  = (TP + TN) / (TP + TN + FP + FN + eps)
    precision = TP / (TP + FP + eps)
    recall    = TP / (TP + FN + eps)
    f1        = 2 * TP / (2 * TP + FP + FN + eps)
    dice      = dice_score(pred, target, threshold, eps).item()
    iou       = iou_score(pred, target, threshold, eps).item()
    return dict(dice=dice, iou=iou, accuracy=accuracy,
                precision=precision, recall=recall, f1=f1)

## Model Architectures

### SMP-based models (U-Net++, DeepLabV3, DeepLabV3+)

In [ ]:
# SMP-based model factory
def build_smp_model(name: str) -> nn.Module:
    """
    Factory for segmentation_models_pytorch models.
    Each architecture is configured exactly as in the paper (Table 1).
    """
    if name == "unetpp":
        return smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=3, classes=1, activation=None)

    if name == "deeplabv3":
        return smp.DeepLabV3(
            encoder_name="resnet34", encoder_weights="imagenet",
            in_channels=3, classes=1, activation=None)

    if name == "deeplabv3plus":
        return smp.DeepLabV3Plus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=3, classes=1, activation=None)

    raise ValueError(f"Unknown SMP model: {name}")

### Attention U-Net

In [ ]:
# Attention U-Net
class AttentionBlock(nn.Module):
    """Standard additive attention gate as in Oktay et al. 2018."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        return x * self.psi(psi)


class AttentionUNet(nn.Module):
    """5-level U-Net with attention gates on every skip connection."""
    def __init__(self):
        super().__init__()

        def cblock(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1), nn.ReLU(inplace=True))

        self.pool = nn.MaxPool2d(2)
        # Encoder
        self.e1 = cblock(3, 64);   self.e2 = cblock(64, 128)
        self.e3 = cblock(128, 256); self.e4 = cblock(256, 512)
        self.e5 = cblock(512, 1024)
        # Decoder
        self.up5 = nn.ConvTranspose2d(1024, 512, 2, 2); self.att5 = AttentionBlock(512, 512, 256); self.d5 = cblock(1024, 512)
        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2);  self.att4 = AttentionBlock(256, 256, 128); self.d4 = cblock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2);  self.att3 = AttentionBlock(128, 128, 64);  self.d3 = cblock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2);   self.att2 = AttentionBlock(64, 64, 32);   self.d2 = cblock(128, 64)
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        x1 = self.e1(x)
        x2 = self.e2(self.pool(x1))
        x3 = self.e3(self.pool(x2))
        x4 = self.e4(self.pool(x3))
        x5 = self.e5(self.pool(x4))

        d = self.up5(x5);  d = torch.cat([self.att5(d, x4), d], 1); d = self.d5(d)
        d = self.up4(d);   d = torch.cat([self.att4(d, x3), d], 1); d = self.d4(d)
        d = self.up3(d);   d = torch.cat([self.att3(d, x2), d], 1); d = self.d3(d)
        d = self.up2(d);   d = torch.cat([self.att2(d, x1), d], 1); d = self.d2(d)
        return self.out(d)

### SegNet

In [ ]:
# SegNet
def _conv_block(in_ch, out_ch, n=2):
    layers = []
    for _ in range(n):
        layers += [nn.Conv2d(in_ch, out_ch, 3, padding=1),
                   nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        in_ch = out_ch
    return nn.Sequential(*layers)


class SegNet(nn.Module):
    """
    VGG-style encoder-decoder with max-pool unpooling (pooling indices reused
    for precise spatial reconstruction — no skip connections).
    """
    def __init__(self, in_ch=3, out_ch=1, ch=[64, 128, 256, 512, 512]):
        super().__init__()
        # Encoder
        self.enc1 = _conv_block(in_ch, ch[0], 2); self.p1 = nn.MaxPool2d(2, 2, return_indices=True)
        self.enc2 = _conv_block(ch[0], ch[1], 2); self.p2 = nn.MaxPool2d(2, 2, return_indices=True)
        self.enc3 = _conv_block(ch[1], ch[2], 3); self.p3 = nn.MaxPool2d(2, 2, return_indices=True)
        self.enc4 = _conv_block(ch[2], ch[3], 3); self.p4 = nn.MaxPool2d(2, 2, return_indices=True)
        self.enc5 = _conv_block(ch[3], ch[4], 3); self.p5 = nn.MaxPool2d(2, 2, return_indices=True)
        # Decoder (mirror)
        self.u5 = nn.MaxUnpool2d(2, 2); self.dec5 = _conv_block(ch[4], ch[3], 3)
        self.u4 = nn.MaxUnpool2d(2, 2); self.dec4 = _conv_block(ch[3], ch[2], 3)
        self.u3 = nn.MaxUnpool2d(2, 2); self.dec3 = _conv_block(ch[2], ch[1], 3)
        self.u2 = nn.MaxUnpool2d(2, 2); self.dec2 = _conv_block(ch[1], ch[0], 2)
        self.u1 = nn.MaxUnpool2d(2, 2); self.dec1 = _conv_block(ch[0], ch[0], 2)
        self.clf = nn.Conv2d(ch[0], out_ch, 1)

    def forward(self, x):
        x1, s1 = self.p1(self.enc1(x));  sz1 = x1.size()
        x2, s2 = self.p2(self.enc2(x1)); sz2 = x2.size()
        # note: sz needed for MaxUnpool to resolve ambiguous sizes
        e1 = self.enc1(x)
        p1, i1 = self.p1(e1)
        e2 = self.enc2(p1)
        p2, i2 = self.p2(e2)
        e3 = self.enc3(p2)
        p3, i3 = self.p3(e3)
        e4 = self.enc4(p3)
        p4, i4 = self.p4(e4)
        e5 = self.enc5(p4)
        p5, i5 = self.p5(e5)

        d = self.dec5(self.u5(p5, i5, output_size=e5.size()))
        d = self.dec4(self.u4(d,  i4, output_size=e4.size()))
        d = self.dec3(self.u3(d,  i3, output_size=e3.size()))
        d = self.dec2(self.u2(d,  i2, output_size=e2.size()))
        d = self.dec1(self.u1(d,  i1, output_size=e1.size()))
        return self.clf(d)

    # override forward to avoid double-encoding
    def forward(self, x):
        e1 = self.enc1(x);  p1, i1 = self.p1(e1)
        e2 = self.enc2(p1); p2, i2 = self.p2(e2)
        e3 = self.enc3(p2); p3, i3 = self.p3(e3)
        e4 = self.enc4(p3); p4, i4 = self.p4(e4)
        e5 = self.enc5(p4); p5, i5 = self.p5(e5)
        d = self.dec5(self.u5(p5, i5, output_size=e5.size()))
        d = self.dec4(self.u4(d,  i4, output_size=e4.size()))
        d = self.dec3(self.u3(d,  i3, output_size=e3.size()))
        d = self.dec2(self.u2(d,  i2, output_size=e2.size()))
        d = self.dec1(self.u1(d,  i1, output_size=e1.size()))
        return self.clf(d)

### RefineNet

In [ ]:
# RefineNet
class CRPBlock(nn.Module):
    """Chained Residual Pooling block — captures multi-scale context."""
    def __init__(self, in_ch, out_ch, n_stages=2):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv2d(in_ch, out_ch, 3, padding=1)
                                    for _ in range(n_stages)])
        self.relu  = nn.ReLU(inplace=False)  # inplace=False avoids in-place autograd issues

    def forward(self, x):
        x    = self.relu(x)
        path = x
        for conv in self.convs:
            path = conv(F.max_pool2d(path, 5, stride=1, padding=2))
            x    = x + path
        return x


class RefineBlock(nn.Module):
    """Single RefineNet block: RCU → CRP → fuse with upsampled residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.rcu    = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=False),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=False))
        self.crp    = CRPBlock(out_ch, out_ch)
        self.proj   = nn.Conv2d(out_ch, out_ch, 1)

    def forward(self, x, residual=None):
        x = self.proj(self.crp(self.rcu(x)))
        if residual is not None:
            x = x + F.interpolate(residual, size=x.shape[2:],
                                  mode="bilinear", align_corners=False)
        return x


class RefineNet(nn.Module):
    """
    ResNet-34 encoder + 4 RefineBlocks for multi-path refinement.
    Best performing segmentation model in the paper (Dice 0.9665).
    """
    def __init__(self):
        super().__init__()
        resnet       = torch.hub.load("pytorch/vision:v0.10.0", "resnet34",
                                      pretrained=True)
        self.layer0  = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.maxpool = resnet.maxpool
        self.layer1  = resnet.layer1   # 64 ch
        self.layer2  = resnet.layer2   # 128 ch
        self.layer3  = resnet.layer3   # 256 ch
        self.layer4  = resnet.layer4   # 512 ch

        self.r4 = RefineBlock(512, 256)
        self.r3 = RefineBlock(256, 256)
        self.r2 = RefineBlock(128, 256)
        self.r1 = RefineBlock(64,  256)
        self.head = nn.Conv2d(256, 1, 1)

    def forward(self, x):
        l0 = self.layer0(x)
        l1 = self.layer1(self.maxpool(l0))
        l2 = self.layer2(l1)
        l3 = self.layer3(l2)
        l4 = self.layer4(l3)

        r4 = self.r4(l4)
        r3 = self.r3(l3, r4)
        r2 = self.r2(l2, r3)
        r1 = self.r1(l1, r2)
        out = self.head(r1)
        return F.interpolate(out, size=x.shape[2:],
                             mode="bilinear", align_corners=False)

### Swin-UNet

In [ ]:
# Swin-UNet
class SwinUNet(nn.Module):
    """
    Swin-Tiny backbone + lightweight conv decoder. Input must be 224×224.
    Extracts the last feature map (B, 768, 7, 7) and upsamples ×32 back to input.
    """
    def __init__(self):
        super().__init__()
        self.backbone = swin_tiny_patch4_window7_224(pretrained=True,
                                                     features_only=True)
        self.decoder  = nn.Sequential(
            nn.Conv2d(768, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 7→14
            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 14→28
            nn.Conv2d(128, 64, 3, padding=1),  nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 28→56
            nn.Conv2d(64, 32, 3, padding=1),   nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 56→112
            nn.Conv2d(32, 16, 3, padding=1),   nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 112→224
            nn.Conv2d(16, 1, 1),
        )

    def forward(self, x):
        feats = self.backbone(x)[-1]           # (B, 768, 7, 7) or (B, 7, 7, 768)
        if feats.shape[1] != 768:
            feats = feats.permute(0, 3, 1, 2)  # NHWC → NCHW
        return self.decoder(feats)

### TransUNet

In [ ]:
# TransUNet
class TransUNet(nn.Module):
    """
    ViT-B/16 encoder + 4-stage CNN decoder. Input must be 224×224.
    Uses patch tokens (196 patches, 768-dim each) reshaped to a spatial feature map.
    """
    def __init__(self):
        super().__init__()
        self.encoder      = vit_base_patch16_224(pretrained=True)
        self.encoder.head = nn.Identity()   # remove classification head

        self.decoder = nn.Sequential(
            nn.Conv2d(768, 384, 3, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 14→28
            nn.Conv2d(384, 192, 3, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 28→56
            nn.Conv2d(192, 96, 3, padding=1),  nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 56→112
            nn.Conv2d(96, 48, 3, padding=1),   nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2),  # 112→224
            nn.Conv2d(48, 1, 1),
        )

    def forward(self, x):
        # Encode patches (skip CLS token from positional embedding)
        tokens = self.encoder.patch_embed(x)                    # (B, 196, 768)
        tokens = tokens + self.encoder.pos_embed[:, 1:, :]      # positional encoding
        tokens = self.encoder.pos_drop(tokens)
        for blk in self.encoder.blocks:
            tokens = blk(tokens)
        tokens = self.encoder.norm(tokens)                       # (B, 196, 768)
        # Reshape to spatial feature map
        fmap = rearrange(tokens, "b (h w) c -> b c h w", h=14, w=14)
        return self.decoder(fmap)

## Training Loop

In [ ]:
# Generic training function
def train_one_model(model, train_loader, val_loader, loss_fn,
                    save_path, use_amp=True):
    """
    Trains a model to completion with:
      - Adam optimiser (lr=1e-4)
      - ReduceLROnPlateau (patience=5, factor=0.5)
      - Early stopping (patience=40 on val Dice)
      - Mixed-precision (AMP) for speed on GPU
      - Checkpoint saved whenever val Dice improves

    Returns a dict of training curves for plotting.
    """
    model     = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode="max",
                                  patience=5, factor=0.5)
    scaler    = GradScaler(enabled=use_amp and DEVICE.type == "cuda")

    best_dice, patience_count = 0.0, 0
    history = dict(train_loss=[], val_loss=[], val_dice=[], val_iou=[])

    for epoch in range(NUM_EPOCHS):
        # Train
        model.train()
        t_loss = 0.0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1:03d}",
                                leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with autocast(device_type=DEVICE.type, enabled=use_amp):
                preds = model(imgs)
                loss  = loss_fn(preds, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            t_loss += loss.item()
        history["train_loss"].append(t_loss / len(train_loader))

        # Validate
        model.eval()
        v_loss = v_dice = v_iou = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                v_loss += loss_fn(preds, masks).item()
                v_dice += dice_score(preds, masks).item()
                v_iou  += iou_score(preds, masks).item()
        n = len(val_loader)
        v_loss /= n; v_dice /= n; v_iou /= n
        history["val_loss"].append(v_loss)
        history["val_dice"].append(v_dice)
        history["val_iou"].append(v_iou)

        scheduler.step(v_dice)
        print(f"  Epoch {epoch+1:03d} | train {history['train_loss'][-1]:.4f} "
              f"| val {v_loss:.4f} | dice {v_dice:.4f} | iou {v_iou:.4f}")

        if v_dice > best_dice:
            best_dice = v_dice
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Saved best (Dice {best_dice:.4f})")
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}.")
                break

    return history

## Run All Models (3 Seeds Each)

Each model is trained 3× with different random seeds and results are averaged. 
Results match Table 4 in the paper.

In [ ]:
# Model registry
# Each entry: (display_name, factory_fn, loss, use_224)
# use_224=True means the model needs 224×224 input (Swin-UNet, TransUNet)

def make_models():
    return [
        ("UNet++",         lambda: build_smp_model("unetpp"),        FocalTverskyLoss(),  False),
        ("DeepLabV3",      lambda: build_smp_model("deeplabv3"),     BCEDiceLoss(),       False),
        ("DeepLabV3+",     lambda: build_smp_model("deeplabv3plus"), FocalTverskyLoss(),  False),
        ("AttentionUNet",  lambda: AttentionUNet(),                   BCEDiceLoss(),       False),
        ("SegNet",         lambda: SegNet(),                          FocalTverskyLoss(),  False),
        ("RefineNet",      lambda: RefineNet(),                       FocalTverskyLoss(),  False),
        ("SwinUNet",       lambda: SwinUNet(),                        nn.BCEWithLogitsLoss(), True),
        ("TransUNet",      lambda: TransUNet(),                       nn.BCEWithLogitsLoss(), True),
    ]

In [ ]:
# Multi-seed training loop
all_results = {}   # model_name → list of per-seed metric dicts

for model_name, model_fn, loss_fn, use_224 in make_models():
    print(f"\n{'='*60}")
    print(f"  Model: {model_name}")
    print(f"{'='*60}")

    seed_metrics = []

    for seed in SEEDS:
        print(f"\n  --- Seed {seed} ---")
        set_seed(seed)

        # Pick correct image size
        t_tfm = train_transform_224 if use_224 else train_transform
        v_tfm = val_transform_224   if use_224 else val_transform

        # Re-build datasets each seed (transform may differ)
        full_ds = SLIMIADataset(CSV_PATH, transform=t_tfm)
        val_ds  = SLIMIADataset(CSV_PATH, transform=v_tfm)

        indices                = list(range(len(full_ds)))
        train_idx, val_idx     = train_test_split(indices, test_size=0.2,
                                                  random_state=seed)
        train_loader = DataLoader(Subset(full_ds, train_idx),
                                  batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=2, pin_memory=True)
        val_loader   = DataLoader(Subset(val_ds, val_idx),
                                  batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=2, pin_memory=True)

        model     = model_fn()
        save_path = os.path.join(CKPT_DIR, f"{model_name}_seed{seed}.pth")

        history = train_one_model(model, train_loader, val_loader,
                                  loss_fn, save_path)

        # Final evaluation on val set with best checkpoint
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        model.eval()
        metrics_acc = {k: 0.0 for k in ["dice","iou","accuracy",
                                         "precision","recall","f1"]}
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                for k, v in compute_all_metrics(preds, masks).items():
                    metrics_acc[k] += v
        n_batches = len(val_loader)
        metrics_avg = {k: v / n_batches for k, v in metrics_acc.items()}
        seed_metrics.append(metrics_avg)
        print(f"  Seed {seed} results: ", 
              {k: f"{v:.4f}" for k, v in metrics_avg.items()})

    all_results[model_name] = seed_metrics

## Results Table

In [ ]:
# Aggregate results (mean ± std across 3 seeds)
rows = []
for model_name, seed_metrics in all_results.items():
    row = {"Model": model_name}
    for metric in ["dice", "iou", "accuracy", "precision", "recall", "f1"]:
        vals = [m[metric] for m in seed_metrics]
        row[metric] = f"{np.mean(vals):.4f} ± {np.std(vals):.5f}"
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("Model")
print("\nSegmentation Results (mean ± std across seeds 42 / 123 / 999)")
print(results_df.to_string())

os.makedirs("../results/tables", exist_ok=True)
results_df.to_csv("../results/tables/segmentation_results.csv")
print("\nSaved to ../results/tables/segmentation_results.csv")

## Visualise Training Curves

In [ ]:
# Quick qualitative check: plot val Dice for all models
# Re-run a single seed to get history (or load from saved history if you cached it)
# This cell is a template — hook up your saved histories as needed.

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
model_names = list(all_results.keys())

for ax, name in zip(axes.flat, model_names):
    # Placeholder — replace with your saved history dicts if running interactively
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Val Dice")
    ax.grid(True, alpha=0.3)

plt.suptitle("Validation Dice Curves — All Segmentation Models", fontsize=13)
plt.tight_layout()
os.makedirs("../results/figures", exist_ok=True)
plt.savefig("../results/figures/segmentation_dice_curves.png", dpi=150)
plt.show()

## Inference on Full Dataset (for Morphometry Extraction)

The best performing model (**RefineNet**) is used to generate segmentation masks
for the entire dataset. These masks are the input to `03_morphometry_extraction.ipynb`.

Ground-truth masks are **never** used during IPP training, only the auto-predicted masks.

In [ ]:
# Run RefineNet inference on the full dataset
BEST_MODEL = "RefineNet"
BEST_SEED  = 42     # or whichever seed gave the best val Dice
BEST_CKPT  = os.path.join(CKPT_DIR, f"{BEST_MODEL}_seed{BEST_SEED}.pth")
PRED_DIR   = "../data/predicted_masks/"
os.makedirs(PRED_DIR, exist_ok=True)

refinenet = RefineNet().to(DEVICE)
refinenet.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
refinenet.eval()

full_ds = SLIMIADataset(CSV_PATH, transform=val_transform)

summary_rows = []
with torch.no_grad():
    for i in tqdm(range(len(full_ds)), desc="Running RefineNet inference"):
        img, mask = full_ds[i]
        img_path  = full_ds.df.iloc[i]["full_path"]

        pred = refinenet(img.unsqueeze(0).to(DEVICE))
        pred_mask = (torch.sigmoid(pred) > 0.5).squeeze().cpu().numpy().astype(np.uint8)

        # Save predicted mask alongside ground truth path
        fname     = os.path.splitext(os.path.basename(img_path))[0] + "_pred.tiff"
        pred_path = os.path.join(PRED_DIR, fname)
        tifffile.imwrite(pred_path, pred_mask)

        # Compute per-image metrics
        m = compute_all_metrics(pred, mask.unsqueeze(0).to(DEVICE))
        m["img_path"]  = img_path
        m["pred_path"] = pred_path
        summary_rows.append(m)

inference_df = pd.DataFrame(summary_rows)
inference_df.to_csv("../data/refinenet_inference_summary.csv", index=False)
print(f"\nInference complete. Mean Dice: {inference_df['dice'].mean():.4f}")
print(f"Predicted masks saved to {PRED_DIR}")